# Comparing centers.csv and numpy grid methods

In [33]:
import pandas as pd
import numpy as np
import tensorflow as tf

grid = pd.read_csv("/home/shilaj/repitframework/repitframework/Assets/natural_convection/centres.csv", header=None)
t_data = np.load("/home/shilaj/repitframework/repitframework/Assets/natural_convection/T_10.01.npy")

grid = grid.drop(0, axis=0).astype(np.float32)
grid_tf = tf.reshape(grid, shape=[-1,4])
grid_np = grid_tf.numpy()
ngrid_xnp = np.array(200*grid_np[:,1:2]+0.51, dtype=np.int64)
ngrid_ynp = np.array(200*grid_np[:,2:3]+0.51, dtype=np.int64)

In [34]:
t_matrix = np.ones([200,200])
for i in range(0, 40000):
    ngrid_x = ngrid_xnp[i]
    ngrid_y = ngrid_ynp[i]
    t_matrix[ngrid_x-1,ngrid_y-1] = t_data[i]

In [35]:
t_matrix_np = t_data.reshape(200,200, order='F')

## Below comparison shows that; to conform it with professor's code we have to order it in F-major order

In [36]:
np.allclose(t_matrix, t_matrix_np)


True

# Comparing professor's method and my method for feature selection:

In [58]:
import numpy as np
from pathlib import Path
import json
import Ofpp
import re
from repitframework import OpenFOAM

In [59]:
data = np.load("/home/shilaj/repitframework/repitframework/Assets/natural_convection/U_10.01.npy")
ux = data[:, 0].reshape(200,200, order="F")
uy = data[:, 1].reshape(200,200, order="F")

In [60]:
# Professors method: 
ux_pad = np.pad(ux, ((1,1), (1,1)), mode="constant", constant_values=0)
prof_method_ux = np.zeros([39204, 5])
k = 0
for i in range(2,200):
	for j in range(2,200):
		prof_method_ux[k,0] = ux_pad[i,j]
		prof_method_ux[k,1] = ux_pad[i-1,j]
		prof_method_ux[k,2] = ux_pad[i+1,j]
		prof_method_ux[k,3] = ux_pad[i,j-1]
		prof_method_ux[k,4] = ux_pad[i,j+1]
		k += 1

In [61]:
# My method:
window_shape = (3, 3)
sliding_window = np.lib.stride_tricks.sliding_window_view(ux, window_shape)
x,y = 1,1
correlated_features = np.stack([
	sliding_window[:,:,x,y],
	sliding_window[:,:,x-1,y],
	sliding_window[:,:,x+1,y],
	sliding_window[:,:,x,y-1],
	sliding_window[:,:,x,y+1]
],axis=-1).reshape(-1,5)

In [65]:
ux

array([[-0.0119128 , -0.00999168, -0.00788239, ...,  0.0246806 ,
         0.0189697 , -0.00594518],
       [-0.0169824 , -0.0160015 , -0.0133442 , ...,  0.066698  ,
         0.0629324 ,  0.0460888 ],
       [-0.0148263 , -0.0170253 , -0.0158739 , ...,  0.112206  ,
         0.112176  ,  0.0991994 ],
       ...,
       [-0.0110955 , -0.0269181 , -0.0382983 , ...,  0.012997  ,
         0.0128753 ,  0.00976527],
       [ 0.0135006 , -0.00434889, -0.0176506 , ...,  0.00971267,
         0.0108962 ,  0.0101799 ],
       [ 0.0108766 ,  0.0096557 ,  0.00640758, ...,  0.00424294,
         0.00531634,  0.00598413]])

In [63]:
prof_method_ux

array([[-0.0160015 , -0.00999168, -0.0170253 , -0.0169824 , -0.0133442 ],
       [-0.0133442 , -0.00788239, -0.0158739 , -0.0160015 , -0.0115032 ],
       [-0.0115032 , -0.00639967, -0.0144757 , -0.0133442 , -0.0101853 ],
       ...,
       [ 0.00876966,  0.0125816 ,  0.00346555,  0.00805921,  0.00971267],
       [ 0.00971267,  0.012997  ,  0.00424294,  0.00876966,  0.0108962 ],
       [ 0.0108962 ,  0.0128753 ,  0.00531634,  0.00971267,  0.0101799 ]])

In [64]:
correlated_features

array([[-0.0160015 , -0.00999168, -0.0170253 , -0.0169824 , -0.0133442 ],
       [-0.0133442 , -0.00788239, -0.0158739 , -0.0160015 , -0.0115032 ],
       [-0.0115032 , -0.00639967, -0.0144757 , -0.0133442 , -0.0101853 ],
       ...,
       [ 0.00876966,  0.0125816 ,  0.00346555,  0.00805921,  0.00971267],
       [ 0.00971267,  0.012997  ,  0.00424294,  0.00876966,  0.0108962 ],
       [ 0.0108962 ,  0.0128753 ,  0.00531634,  0.00971267,  0.0101799 ]])

In [62]:
np.allclose(prof_method_ux, correlated_features)

True

# Comparing normalization and denormalization using numpy and pandas

In [29]:
import numpy as np
import pandas as pd

In [28]:
ml_dataset = np.random.rand(39204*2, 18)
dataset = ml_dataset[:,3:18]
train_labels = ml_dataset[:,0:3]

In [38]:
# Normalize the dataset using pandas:
dataset_df = pd.DataFrame(dataset)
train_stats = dataset_df.describe().transpose()

labels_df = pd.DataFrame(train_labels)
labels_stats = labels_df.describe().transpose()

In [40]:
normed_dataset = (dataset_df - train_stats['mean']) / train_stats['std']
normed_labels = (labels_df - labels_stats['mean']) / labels_stats['std']
unnormed_labels = (normed_labels * labels_stats['std']) + labels_stats['mean']

In [41]:
# Normalize the dataset using numpy:
mean_train = np.mean(dataset, axis=0)
std_train  = np.std(dataset, axis=0)

mean_labels = np.mean(train_labels, axis=0)
std_labels = np.std(train_labels, axis=0)

normed_dataset_np = (dataset - mean_train) / std_train
normed_labels_np = (train_labels - mean_labels) / std_labels
unnormed_labels_np = (normed_labels_np * std_labels) + mean_labels

In [47]:
np.allclose(normed_dataset, normed_dataset_np)

True

In [48]:
np.allclose(normed_labels, normed_labels_np)

True

In [51]:
np.allclose(unnormed_labels, unnormed_labels_np)

True

In [53]:
np.allclose(unnormed_labels, train_labels)

True

# Comparing difference calculation using framework and numpy

In [1]:
import numpy as np

from repitframework.OpenFOAM.utils import OpenfoamUtils
from repitframework.config import TrainingConfig
from repitframework.Dataset.fvmn import FVMNDataset

In [2]:
training_config = TrainingConfig()
fvmn_dataset = FVMNDataset(training_config)
t_1002 = np.load("/home/shilaj/repitframework/repitframework/Assets/natural_convection/T_10.02.npy")
t_1003 = np.load("/home/shilaj/repitframework/repitframework/Assets/natural_convection/T_10.03.npy")
u_1002 = np.load("/home/shilaj/repitframework/repitframework/Assets/natural_convection/U_10.02.npy")
u_1003 = np.load("/home/shilaj/repitframework/repitframework/Assets/natural_convection/U_10.03.npy")

In [3]:
input_data = fvmn_dataset._prepare_input(10.02)

In [13]:
t_framework_internal = input_data[:,::5][:,2:]

In [14]:
t_1002_internal = t_1002.reshape(200,200, order='F')[1:-1,1:-1].reshape(-1,1, order='F')

In [15]:
np.allclose(t_framework_internal, t_1002_internal)

True

In [16]:
# Load T value for time step 10.02 and 10.03:
t_now = np.load("/home/shilaj/repitframework/repitframework/Assets/natural_convection/T_10.01.npy").reshape(200,200, order="F")[1:-1,1:-1].reshape(-1,1, order="F")
t_next = np.load("/home/shilaj/repitframework/repitframework/Assets/natural_convection/T_10.02.npy").reshape(200,200, order="F")[1:-1,1:-1].reshape(-1,1, order="F")
u_now = np.load("/home/shilaj/repitframework/repitframework/Assets/natural_convection/U_10.01.npy")
ux_now = u_now[:,0].reshape(200,200, order="F")[1:-1,1:-1].reshape(-1,1, order="F")
uy_now = u_now[:,1].reshape(200,200, order="F")[1:-1,1:-1].reshape(-1,1, order="F")
U_next = np.load("/home/shilaj/repitframework/repitframework/Assets/natural_convection/U_10.02.npy")
ux_next = U_next[:,0].reshape(200,200, order="F")[1:-1,1:-1].reshape(-1,1, order="F")
uy_next = U_next[:,1].reshape(200,200, order="F")[1:-1,1:-1].reshape(-1,1, order="F")

data_now = np.concatenate([ux_now,uy_now, t_now], axis=1)
data_next = np.concatenate([ux_next, uy_next, t_next], axis=1)
data_diff_np = data_next - data_now

In [17]:
data_diff_framework = fvmn_dataset._calculate_difference(time=10.01)

In [20]:
np.allclose(data_diff_np, data_diff_framework)

True

# Run the prediction 

In [29]:
from repitframework.Models.FVMN.fvmn import FVMNetwork
from repitframework.config import TrainingConfig, OpenfoamConfig
from repitframework.Dataset.fvmn import FVMNDataset
from runner import Trainer

In [31]:
training_config = TrainingConfig()
model = FVMNetwork(training_config)
trainer = Trainer(training_config,model, training_config.optimizer,
                  training_config.loss, model_name="best_model.pth")

In [32]:
trainer.predict()

10.03